> Historical development experiment. Read [notebook20](20_practical_options_report.ipynb) for the final recommendation.

# Drone-video test: animal gate works, context needs verification

**The combined prototype is not ready to promote.** The aerial test exposed context errors that the animal-presence gate cannot catch: a sea lion was proposed as a person, and a rock carrying pinnipeds was proposed as a vehicle.

Four newly reviewed aerial clips, 24 sampled frames each (first 12 seconds at 2 FPS), with the previous settings frozen. This is aerial stock footage, not a DJI field recording.

| Clip | Animal gate active | Person frames | Vehicle frames |
|---|---:|---:|---:|
| Platform | 24/24 | 0 | 0 |
| Surf | 13/24 | 2 | 0 |
| Rocks | 24/24 | 0 | 8 |
| Beach control | 0/24 | 0 | 0 |

**Gate-open/display counts are not accuracy, recall or unique-object counts.** Surf visibility varies with wave occlusion. The two unexpected context types above were visually checked at native-resolution crops; their examples are shown below.

**Main trial: 96 frames, 32 Gemini requests, zero failed requests.** All 12 integrity/state-replay checks and 23 policy fixtures passed. Both detector arms shared the same DMD animal proposals and actual Gemini decisions. Full baseline/combined counts are in report.json.

## Two failure modes and the next prototype

1. **Animal presence does not validate context boxes.** A separate, post-hoc three-request crop check rejected the inspected false person and vehicle proposals, while retaining two person proposals and one truck proposal in the earlier ground-level seal-release frame. These five proposals across three selected frames are a diagnostic, not held-out evidence or full-video context filtering. This check has not been applied to the 96-frame replays.
2. **Stale evidence survives a hard cut.** An explicitly synthetic cut from platform animals to the actual beach-control frames left context displayed for six control frames (source offsets 0–2.5025 seconds) without a cut reset. Providing the known scene reset reduced this to zero. Automatic cut detection is not implemented; this checks the reset hook, not natural animal exits.

The next candidate should verify **both animal evidence and the proposed person/vehicle crops**, and clear evidence on reliable scene transitions. It needs new aerial co-occurrence validation before promotion.

## Timing is still a limitation

The 4K tiled DMD pass took roughly **2.0 seconds per analyzed frame** on the Tesla T4. The 1080p clips took about 0.67–0.68 seconds. YOLO-World context took about 0.03–0.06 seconds; individual Gemini requests had medians of about 2.6–3.7 seconds. The 2-FPS replay therefore does not demonstrate a live 2-FPS pipeline.

## What the overlays mean

Yellow boxes show the raw top-three animal proposals selected for review, including false proposals. Gemini verifies animal presence, not the correctness of every yellow box. Cyan person and orange vehicle proposals are shown only while the animal-evidence gate is active. A platform is not automatically a vehicle, and birds are outside this DMD animal ontology.

## Coverage and remaining gap

Platform and rocky-islet screening showed clear pinnipeds; the surf animal is small and intermittently obscured by waves. The crowded-beach screening showed people/umbrellas with no mammal identified, but absence is not exhaustively annotated.
**No verified marine-animal-plus-human/vehicle aerial positive was found in these clips.** This trial tests aerial animal evidence and false context activation; it cannot establish marine co-occurrence retention. It is neither DJI capture validation nor a rescue-need classifier.

## Fixed settings and timing

DMD full-frame + native 1280 tiles, 100-pixel overlap, score 0.3 and merge IoU 0.5. YOLO-World full-frame context uses the frozen 21-name vocabulary. Gemini reviews at samples 0, 6, 12 and 18: max three native crops plus overview, and a separate scene check. Gate accepts same-check agreement or two consecutive scene confirmations within four seconds; same-scene evidence hold is three seconds. Both reviewers use the same Gemini model and may hallucinate together.
Source-time replay applies completed reviews at their source timestamps. Measured API/detector timing is archived, but this is not real-time streaming throughput. Automatic scene-cut detection and natural disappearance timing remain untested; known source changes and expiry are covered by state replay/fixtures.

## Sources

- [raft](https://www.pexels.com/video/aerial-view-of-seals-resting-on-raft-40906785/) — Marc Espejo/Pexels; 3840×2160, 18.35s, fully decoded. Derivative overlays are sampled/resized.
- [surf](https://www.pexels.com/video/a-sea-lion-swimming-along-the-waves-in-a-beach-3006844/) — Athena Sandrini/Pexels; 1920×1080, 15.04s, fully decoded. Derivative overlays are sampled/resized.
- [rocks](https://www.pexels.com/video/aerial-view-of-seals-resting-on-ocean-rocks-30824194/) — Lost Souls NV/Pexels; 1920×1080, 17.38s, fully decoded. Derivative overlays are sampled/resized.
- [crowded_beach](https://www.pexels.com/video/aerial-view-of-beach-with-people-and-umbrellas-32726982/) — 8 K/Pexels; 3840×2160, 18.94s, fully decoded. Derivative overlays are sampled/resized.

No model weights/prompts changed, no production integration or deployment, and no automatic incident/dispatch.

![Research evidence](../assets/surf-false-person-crop.jpg)

Attribution: [CREDITS.json](../assets/CREDITS.json).

![Research evidence](../assets/rock-false-vehicle-crop.jpg)

Attribution: [CREDITS.json](../assets/CREDITS.json).

In [ ]:
from pathlib import Path
import json
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'research/aerial_detector/results').is_dir())
research=root/'research/aerial_detector'
print(json.dumps(json.loads((research/'results/aerial-patrol-v1/report.json').read_text()),indent=2))